# RQ1: transfer — training

How the transfer of continued pretraining (CPT) develops during training: on held-out credit tables, on the run's own training tables and on the non-credit retention panel (E1), how much of it is training randomness (E2: seeds 43 and 44 next to E1's seed 42), and whether CPT on matched non-credit tables develops the same way (E3).

Monitor points use one fixed context/query split per table (up to 10,000 rows, two preprocessing views); they show how the effect develops and are not the five-fold evaluation.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pandas as pd
from src.visualize import style, campaign as cp, campaign2 as c2, results2 as r2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('rq1/01_training')
report = cp.NotebookReport('RQ1: transfer — training', sink=sink)
frame = c2.load_trajectories(['reference', 'seed', 'domain_control'])
changes = c2.monitor_changes(frame)
curves = c2.monitor_curves(changes)

## 1. Design

The arms of this research question: what each changes versus the reference recipe, the tasks and the number of training runs.

In [2]:
report.table('Arms of RQ1', c2.design_table(1))
report.add('1. Design', 'Each arm changes only the listed settings; E1 is the shared control of E2, E3 and E5-E9.')

,experiment,slug,arm,track,change,models,runs,tasks
0,E1,reference,0,pd,reference,4.0,16,NaN
1,E1,reference,0,lgd,reference,4.0,16,NaN
2,E2,seed,0,pd,seed=43,4.0,16,NaN
3,E2,seed,1,pd,seed=44,4.0,16,NaN
4,E2,seed,0,lgd,seed=43,4.0,16,NaN
5,E2,seed,1,lgd,seed=44,4.0,16,NaN
6,E3,domain_control,0,pd,corpus=domain_control,4.0,16,NaN
7,E3,domain_control,0,lgd,corpus=domain_control,4.0,16,NaN
8,E4,target_finetune,—,NaN,Is CPT worth it compared with fine-tuning on t...,NaN,0,"PD, LGD"


## 2. How transfer develops, and how much is seed noise (E1, E2)

Question: does the held-out credit score move during CPT, does it track the fit to the training tables, and what happens to the non-credit tables meanwhile? Each curve is an equal-table mean of the per-table change from update 0; the band spans the three reference seeds.

In [3]:
reference = c2.select(curves, slug='reference')
band = c2.seed_band(changes)
fig = r2.monitor_development(reference, band, title='E1 reference: monitored change during continued pretraining')
cp.show(sink, c2.pages(fig, 'reference_monitor',
    'Change of the monitored score from update 0 during continued pretraining with the reference recipe (E1, seed 42), '
    'for PD (AUC, top) and LGD (R², bottom) and each base model (columns). Lines are equal-table means over the held-out '
    'credit tables (solid; each table once across the four partitions), the run\'s own training tables (dashed; each '
    'table averaged over the partitions that train on it) and the non-credit retention panel (dotted; 19 binary or 12 '
    'regression tables, averaged over partitions). Shading spans the held-out credit means of seeds 42 (E1), 43 and 44 '
    '(E2) where all three were monitored. Every point uses one fixed context/query split of up to 10,000 rows per table '
    'and two preprocessing views; it is the training monitor, not the five-fold evaluation.'))
report.table('Change at the last monitor point (E1 and E2)', c2.curve_endpoints(c2.select(curves, slug=['reference', 'seed'])))
report.table('Seed range of the held-out credit mean (E1 and E2)', c2.curve_endpoints(band))
report.add('2. How transfer develops, and how much is seed noise (E1, E2)', c2.evidence_note('E1 monitor curves', reference))

No measurements available for this section yet.


""


""


## 3. Credit versus matched non-credit CPT (E3)

Question: does CPT on the matched non-credit twins of the same training tables (same number of tables, matched rows, features and categorical features) move the held-out credit score as credit CPT does?

In [4]:
control = c2.select(curves, slug=['reference', 'domain_control'], scope='test')
arms = r2.arm_styles(pd.concat([c2.arm_settings('reference', 'pd'), c2.arm_settings('domain_control', 'pd')]), ordered=False)
fig = r2.arm_trajectories([dict(curves=c2.select(control, track='pd'), x='successful_updates', ylabel='PD: Δ AUC'),
                           dict(curves=c2.select(control, track='lgd'), x='successful_updates', ylabel='LGD: Δ R²')],
                          arms, title='Held-out credit monitor: CPT on credit tables (E1) versus their non-credit twins (E3)')
cp.show(sink, c2.pages(fig, 'domain_control_monitor',
    'Change of the equal-table mean held-out credit monitor score from update 0, for CPT on the partition\'s credit '
    'training tables (E1, black) and on their matched non-credit twins (E3), per task (rows) and base model (columns). '
    'Both arms use the reference recipe and are scored on the same held-out credit tables with one fixed context/query '
    'split of up to 10,000 rows per table; this is the training monitor, not the five-fold evaluation.'))
report.table('Held-out credit change at the last monitor point (E1, E3)', c2.curve_endpoints(control))
report.add('3. Credit versus matched non-credit CPT (E3)', c2.evidence_note('E3 monitor curves', c2.select(control, slug='domain_control')))

No measurements available for this section yet.


""


## 4. Training cost

Wall-clock hours per run (with and without monitoring), rows processed and peak reserved GPU memory, as recorded at the last monitor point of each run.

In [5]:
report.table('Training cost per arm and model (E1, E2, E3)', c2.resource_summary(c2.run_resources(frame)))
report.add('4. Training cost', c2.evidence_note('training cost records', frame))

""


## Summary

Every displayed table and figure, in section order.

In [6]:
print(report.summary(sink))

RQ1: transfer — training

1. Design
Each arm changes only the listed settings; E1 is the shared control of E2, E3 and E5-E9.

Table: Arms of RQ1
CSV (first column: row index)
row,experiment,slug,arm,track,change,models,runs,tasks
0,E1,reference,0,pd,reference,4,16,unavailable
1,E1,reference,0,lgd,reference,4,16,unavailable
2,E2,seed,0,pd,seed=43,4,16,unavailable
3,E2,seed,1,pd,seed=44,4,16,unavailable
4,E2,seed,0,lgd,seed=43,4,16,unavailable
5,E2,seed,1,lgd,seed=44,4,16,unavailable
6,E3,domain_control,0,pd,corpus=domain_control,4,16,unavailable
7,E3,domain_control,0,lgd,corpus=domain_control,4,16,unavailable
8,E4,target_finetune,—,unavailable,"Is CPT worth it compared with fine-tuning on the target table, and do the two stack?",unavailable,0,"PD, LGD"

2. How transfer develops, and how much is seed noise (E1, E2)
No results yet for E1 monitor curves; the figure is drawn once they exist.

No measurements available for this figure group.

Table: Change at the last monitor point (E1 and E